# Simple: Cell Suppression with PAMOLA.CORE

**Goal**: Learn cell suppression basics in 5 minutes using operation.execute()

**What you'll learn:**
- Load sample data from examples/data_examples/
- Replace individual cell values using execute()
- Compare before/after value distributions
- Understand fine-grained privacy protection at cell level

**Prerequisites:**
- Python 3.8+
- PAMOLA.CORE installed
- Basic pandas knowledge

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── anonymization/
        └── 01_cell_suppression_simple.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import json
from pathlib import Path
from datetime import datetime
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print(f"   Revision: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.anonymization.suppression.cell_op import CellSuppressionOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Sample Data

**How to use:**
- Run to load from `examples/data_examples/sample.csv`
- Auto-creates 100 patient records if file missing
- Review dataset structure and identify outliers

**What you'll see:**
- Dataset summary (100 records, 8 columns)
- First 5 records with patient measurements
- Column statistics (types, unique counts, ranges)
- Temperature field with outliers (95-105°F vs normal 97-100°F)

**Note:** Auto-generated data includes realistic outliers for suppression demonstration

In [ ]:
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'sample.csv'

if not data_path.exists():
    print("⚠️  File not found, creating sample data...")
    data_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Create realistic patient measurement data with outliers
    np.random.seed(42)
    
    # Normal temperature readings (98.6°F ± 1°F)
    temperatures = np.random.normal(98.6, 1.0, 95)
    # Add some outliers
    outlier_temps = [103.5, 95.2, 104.1, 94.8, 105.2]
    all_temps = np.concatenate([temperatures, outlier_temps])
    
    sample_data = pd.DataFrame({
        'patient_id': [f'P{i:04d}' for i in range(1, 101)],
        'age': np.random.randint(18, 85, 100),
        'temperature': all_temps,
        'blood_pressure_systolic': np.random.randint(100, 140, 100),
        'blood_pressure_diastolic': np.random.randint(60, 90, 100),
        'heart_rate': np.random.randint(60, 100, 100),
        'diagnosis': np.random.choice(['Healthy', 'Flu', 'Cold', 'Infection', 'Other'], 100),
        'measurement_date': pd.date_range('2024-01-01', periods=100, freq='D')
    })
    
    # Shuffle to randomize outlier positions
    sample_data = sample_data.sample(frac=1, random_state=42).reset_index(drop=True)
    sample_data.to_csv(data_path, index=False)
    print(f"✓ Sample data created with outliers")

df = read_csv(data_path)

print(f"📊 Dataset loaded: {len(df)} records, {len(df.columns)} columns")
print(f"\n🔍 First 5 records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:30s} ({dtype_str:20s}): {df[col].nunique()} unique values")
    elif pd.api.types.is_datetime64_any_dtype(df[col]):
        print(f"  {col:30s} ({dtype_str:20s}): {df[col].min()} to {df[col].max()}")
    elif pd.api.types.is_numeric_dtype(df[col]):
        print(f"  {col:30s} ({dtype_str:20s}): min={df[col].min():.2f}, max={df[col].max():.2f}, mean={df[col].mean():.2f}")

## Step 3: Setup Task Environment

**How to use:**
1. **CUSTOMIZE field_name**: Edit `create_config_kwargs()` function
   - Default: `"email"`
   - Change to YOUR target field (e.g., "temperature")
2. Run to validate and setup environment

**What you'll see:**
- Field validation (type, non-null count, unique values)
- Statistics (mean, std, min, max for numeric fields)
- Task directory, TaskReporter, DataSource initialized (✓)

**Note:** Field must be numeric for outlier detection to work properly

In [ ]:
# Configuration helper function (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "field_name": "email"
    }

kwargs = create_config_kwargs()
field_name = kwargs["field_name"]

# Validate field exists
print(f"\n🔍 Validating field configuration...\n")
if field_name not in df.columns:
    raise ValueError(
        f"❌ Field '{field_name}' not found in dataset!\n"
        f"Available columns: {', '.join(df.columns)}\n"
        f"Please update 'field_name' in create_config_kwargs()"
    )

# Function to print safe statistics
def safe_stats(col, label=""):
    """Print safe statistics for numeric, datetime, and non-numeric columns."""
    print(f"\n  {label}:")
    print(f"    Count: {col.notna().sum()}")

    # ==========================================================
    # CASE 1 — NUMERIC
    # ==========================================================
    if pd.api.types.is_numeric_dtype(col):
        desc = col.describe(percentiles=[0.25, 0.5, 0.75])

        print(f"    Mean:  {desc['mean']:.2f}")
        print(f"    Std:   {desc['std']:.2f}")
        print(f"    Min:   {desc['min']:.2f}")
        print(f"    25%:   {desc['25%']:.2f}")
        print(f"    50%:   {desc['50%']:.2f}")
        print(f"    75%:   {desc['75%']:.2f}")
        print(f"    Max:   {desc['max']:.2f}")
        return

    # ==========================================================
    # CASE 2 — DATETIME
    # ==========================================================
    if pd.api.types.is_datetime64_any_dtype(col):
        print(f"    Min:   {col.min()}")
        print(f"    50%:   {col.quantile(0.5)}")
        print(f"    Max:   {col.max()}")
        print(f"    Mean:  {col.mean()}")
        return

    # ==========================================================
    # CASE 3 — NON-NUMERIC (string, category, object)
    # ==========================================================
    print(f"    (non-numeric field)")
    vc = col.value_counts().head(5).to_string()
    print(f"    Top values:")
    print("\n".join([f"      {line}" for line in vc.split("\n")]))

col = df[field_name]

print(f"✓ Field to process: '{field_name}'")
print(f"  Data type: {col.dtype}")
print(f"  Non-null values: {col.notna().sum()}")
print(f"  Unique values: {col.nunique(dropna=True)}")

# 👉 Use safe_stats for all types
safe_stats(col, label="Statistics")

# Create task directory
task_dir = examples_dir / 'data_examples' / 'simple_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="simple_cell_001",
    task_type="cell_suppression",
    description="Cell suppression for outlier field values",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

execute_kwargs = {"dataset_name": kwargs["dataset_name"]}
print(f"✓ Config kwargs: {execute_kwargs}")

# Create DataSource
data_source = DataSource(dataframes={"main_dataset": df})
print("✓ DataSource created")

# Setup progress tracker
tracker = HierarchicalProgressTracker(
    total=6,
    description=f"Suppressing outlier cells in {field_name}",
    unit="steps",
    track_memory=True,
    level=0,
    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"
)
print("✓ Progress tracker ready")

print(f"\n✅ Environment setup complete!")

## Step 4: Configure & Execute Operation

**How to use:**
- Review configuration summary
- Run to execute cell suppression
- Monitor progress bar (6 steps: validate → detect → suppress → metrics → viz → save)

**Key parameters:**
- `suppression_strategy='null'`: Replace outliers with null/NaN
- `suppress_if='outlier'`: Trigger on outlier detection
- `outlier_method='iqr'`: Use IQR method (1.5 × IQR)
- `mode='REPLACE'`: Modify original field

**What you'll see:**
- Configuration summary with all parameters
- Progress bar through 6 processing steps
- Completion status with artifact count (4-6 files)
- Success message (✅ Operation completed!)

**Note:** Execution takes ~2-5 seconds for 100 records

In [ ]:
# Create operation with production-style configuration
operation = CellSuppressionOperation(
    # Core parameters
    field_name=field_name,
    mode='REPLACE',
    
    # Output configuration
    column_prefix='_',
    output_field_name=None,
    output_format='csv',
    
    # Cell suppression strategy
    suppression_strategy='null',          # 'null', 'mean', 'median', 'mode', 'constant', 'group_mean', 'group_mode'
    suppression_value=None,               # Used with 'constant' strategy
    
    # Automatic suppression trigger
    suppress_if='outlier',                # 'outlier', 'rare', 'null', or None
    
    # Outlier detection parameters
    outlier_method='iqr',                 # 'iqr' or 'zscore'
    outlier_threshold=1.5,                # IQR multiplier (1.5 is standard)
    
    # Rare value detection parameters
    rare_threshold=10,                    # Frequency threshold for rare values
    
    # Group-based suppression parameters (not used in this simple example)
    group_by_field=None,                  # Field for grouping (e.g., 'diagnosis')
    min_group_size=5,                     # Minimum group size
    
    # Processing settings
    use_vectorization=False,
    parallel_processes=1,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Operation configured")
print(f"  Field:                {operation.field_name}")
print(f"  Suppression strategy: {operation.suppression_strategy}")
print(f"  Suppress if:          {operation.suppress_if}")
print(f"  Outlier method:       {operation.outlier_method}")
print(f"  Outlier threshold:    {operation.outlier_threshold}")
print(f"  Visualizations:       {operation.generate_visualization}")
print(f"  Save output:          {operation.save_output}")
print(f"  Force recalc:         {operation.force_recalculation}")

# Execute operation
print("\n" + "=" * 80)
print("⚙️  Executing Cell Suppression Operation...")
print("=" * 80 + "\n")

result = operation.execute(
    data_source=data_source,
    task_dir=task_dir,
    reporter=task_reporter,
    progress_tracker=tracker,
    **execute_kwargs
)

print("\n" + "=" * 80)
print(f"✅ Operation completed!")
print(f"   Status: {result.status}")
print(f"   Artifacts: {len(result.artifacts)}")
print("=" * 80)

## Step 5: Analyze Results

**How to use:**
- Run to load and compare processed data
- Review cell suppression statistics

**What you'll see:**
- Artifacts list (CSV, JSON, PNG files)
- Sample records (first 10) with suppressed cells
- Cell suppression analysis (total, suppressed, rate, unchanged)
- Suppressed vs replacement value statistics
- Field distribution comparison (original vs after)

**Note:** Suppression rate typically 5-10% for outlier detection (only extreme values affected). Most data remains unchanged

In [ ]:
# Display generated artifacts
if result.artifacts:
    print("\n📦 Generated Artifacts:")
    print("=" * 80)
    for artifact in result.artifacts:
        print(f"   [{artifact.category}] {artifact.artifact_type}: {artifact.path.name}")

# Find and load output file
output_files = list(task_dir.glob('output/*.csv'))
if output_files:
    output_file = output_files[0]
    result_df = pd.read_csv(output_file)

    # Fix datetime columns
    if pd.api.types.is_datetime64_any_dtype(df[field_name]):
        result_df[field_name] = pd.to_datetime(result_df[field_name])

    print("\n" + "=" * 80)
    print("📊 RESULTS COMPARISON")
    print("=" * 80)

    # Show sample
    print("\n🔍 Sample Processed Records:")
    display_cols = [field_name] + [col for col in df.columns if col != field_name]
    print(result_df[display_cols].head(10))

    # Identify suppressed cells
    suppressed_mask = df[field_name] != result_df[field_name]
    cells_suppressed = suppressed_mask.sum()

    print(f"\n📈 Cell Suppression Analysis:")
    print(f"  Total cells:        {len(df)}")
    print(f"  Cells suppressed:   {cells_suppressed}")
    print(f"  Suppression rate:   {(cells_suppressed / len(df)) * 100:.2f}%")
    print(f"  Cells unchanged:    {len(df) - cells_suppressed}")

    if cells_suppressed > 0:
        suppressed_original = df.loc[suppressed_mask, field_name]
        suppressed_after = result_df.loc[suppressed_mask, field_name]

        safe_stats(suppressed_original, label="Suppressed values (original)")
        safe_stats(suppressed_after, label="Replacement values")

    # Field distribution comparison
    print(f"\n📊 Field Distribution Comparison:")
    safe_stats(df[field_name], label="Original distribution")
    safe_stats(result_df[field_name], label="After suppression")

    print("\n" + "=" * 80)
    print("✨ SUMMARY")
    print("=" * 80)
    print(f"  Original records:   {len(df)}")
    print(f"  Processed records:  {len(result_df)}")
    print(f"  Records unchanged:  {len(df) == len(result_df)}")

    # Display result metrics
    if result.metrics:
        print("\n📊 Key Metrics:")
        for key, value in list(result.metrics.items())[:20]:
            if isinstance(value, (int, float)):
                print(f"  {key:40s}: {value:.4f}" if isinstance(value, float)
                      else f"  {key:40s}: {value}")
            elif isinstance(value, dict):
                print(f"  {key}:")
                for subkey, subval in value.items():
                    print(f"    {subkey:35s}: {subval}")

    print(f"\n💡 Cell Suppression Insight:")
    print(f"   Outlier or sensitive field values have been replaced safely.")
else:
    print("⚠️  No output file found in", task_dir / 'output')

## Step 6: Review Artifacts Location

**How to use:**
- Run to display all generated files
- Navigate to directories for manual inspection
- Verify each folder has expected file count

**What you'll see:**
- Directory structure tree (output/, metrics/, visualizations/, config/)
- File count per directory (typically 1-3 files each)
- File names with sizes in KB
- Absolute path to task directory for manual navigation

**Output structure:**
```
examples/data_examples/simple_output/
├── output/           # Processed CSV (outliers suppressed)
├── metrics/          # Suppression statistics JSON
├── visualizations/   # Before/after comparison charts
└── config/           # Operation configuration
```

**Note:** Copy absolute path to file explorer for manual browsing

In [ ]:
print("\n📂 OUTPUT DIRECTORY STRUCTURE:")
print("=" * 80)
print(f"\nTask directory: {task_dir.absolute()}\n")

for subdir in ['output', 'metrics', 'visualizations', 'config']:
    path = task_dir / subdir
    if path.exists():
        files = list(path.glob('*'))
        print(f"📁 {subdir}/ ({len(files)} files)")
        for f in files[:5]:
            size_kb = f.stat().st_size / 1024
            print(f"   • {f.name} ({size_kb:.1f} KB)")

print("\n✅ All artifacts saved successfully!")

## Step 7: Detailed Artifact Review

**How to use:**
- Run for comprehensive artifact inspection
- Auto-loads NEWEST files from each category
- Focus on validation points in output

**What you'll see:**
1. **Metrics JSON**: Suppression strategy, cells suppressed, rate, outlier detection details
2. **Output Data**: First 10 rows, distribution comparison, statistics
3. **Visualizations**: Before/after comparison charts
4. **Suppression Details**: Original → suppressed values list

**Validate:**
- ✅ Outliers detected and suppressed (5-10% typical)
- ✅ Suppression strategy applied correctly
- ✅ Field distribution normalized (outliers removed)
- ✅ Majority of data unchanged

**Note:** Only newest files shown. Multiple runs create versions - older artifacts excluded automatically

In [ ]:
print("\n📊 DETAILED ARTIFACT REVIEW")
print("=" * 80)

# 1. METRICS JSON (NEWEST)
print("\n1️⃣ METRICS (JSON):")
print("-" * 80)

metrics_dir = task_dir / 'metrics'
if not metrics_dir.exists():
    print(f"⚠️  Metrics directory not found: {metrics_dir}")
else:
    metrics_files = list(metrics_dir.glob('*_metrics_*.json'))
    
    if not metrics_files:
        print("⚠️  No metrics files found")
    else:
        # 1) Exclude data_types inside IF
        filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

        if filtered:
            # Use only filtered files
            target_files = filtered
            print(f"✓ Found {len(filtered)} filtered metrics file(s) (excluded data_types)")
        else:
            # Fallback: nothing left after filtering → use all
            target_files = metrics_files
            print(f"⚠ No filtered metrics found → fallback to ALL {len(metrics_files)} file(s)")

        # 2) Pick latest
        target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_metrics_file = target_files[0]

        print(f"📄 Loading LATEST: {latest_metrics_file.name}")
        mtime = datetime.fromtimestamp(latest_metrics_file.stat().st_mtime)
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_metrics_file.stat().st_size / 1024:.1f} KB\n")
        print("=" * 80)
        
        try:
            with open(latest_metrics_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # Extract metrics from nested structure
            metrics = data.get('metrics', data)
            
            # Display metadata if available
            if 'metadata' in data:
                print("📋 METADATA:")
                metadata = data['metadata']
                print(f"   Timestamp:   {metadata.get('timestamp', 'N/A')}")
                print(f"   Operation:   {metadata.get('name', 'N/A')}")
                if 'operation' in metadata:
                    op = metadata['operation']
                    print(f"   Class:       {op.get('class', 'N/A')}")
                    print(f"   Function:    {op.get('function', 'N/A')}")
            
            # Display cell suppression metrics
            print("\n🔒 CELL SUPPRESSION METRICS:")
            print(f"   Operation Type:          {metrics.get('operation_type', 'N/A')}")
            print(f"   Suppression Strategy:    {metrics.get('suppression_strategy', 'N/A')}")
            print(f"   Cells Suppressed:        {metrics.get('cells_suppressed', 0):,}")
            print(f"   Suppression Rate:        {metrics.get('suppression_rate', 0):.2f}%")
            print(f"   Total Cells Processed:   {metrics.get('total_cells_processed', 0):,}")
            print(f"   Non-null Cells:          {metrics.get('non_null_cells_processed', 0):,}")
            
            # Display outlier detection metrics if applicable
            if 'outlier_method' in metrics:
                print("\n📊 OUTLIER DETECTION:")
                print(f"   Method:                  {metrics.get('outlier_method', 'N/A')}")
                print(f"   Threshold:               {metrics.get('outlier_threshold', 0):.2f}")
                print(f"   Outliers Detected:       {metrics.get('outliers_detected', 0):,}")
            
            # Display rare value metrics if applicable
            if 'rare_threshold' in metrics:
                print("\n📊 RARE VALUE DETECTION:")
                print(f"   Threshold:               {metrics.get('rare_threshold', 0)}")
                if 'rare_values_detected' in metrics:
                    print(f"   Rare Values Detected:    {metrics.get('rare_values_detected', 0):,}")
            
            # Display suppression by reason
            if 'suppression_by_reason' in metrics:
                suppression_reasons = metrics['suppression_by_reason']
                if suppression_reasons:
                    print("\n📋 SUPPRESSION BY REASON:")
                    for reason, count in suppression_reasons.items():
                        print(f"   {reason.capitalize():20s}: {count:,}")
            
            # Display suppression by strategy
            if 'suppression_by_strategy' in metrics:
                suppression_strategies = metrics['suppression_by_strategy']
                if suppression_strategies:
                    print("\n📋 SUPPRESSION BY STRATEGY:")
                    for strategy, count in suppression_strategies.items():
                        print(f"   {strategy.capitalize():20s}: {count:,}")
            
            # Display performance metrics
            if 'duration_seconds' in metrics:
                print("\n⚡ PERFORMANCE:")
                print(f"   Duration:                {metrics.get('duration_seconds', 0):.4f}s")
                print(f"   Records Processed:       {metrics.get('records_processed', 0):,}")
                print(f"   Records/Second:          {metrics.get('records_per_second', 0):.2f}")
                if 'chunk_count' in metrics:
                    print(f"   Chunk Count:             {metrics.get('chunk_count', 0)}")
            
            # Display effectiveness metrics
            temp_eff_ratio = metrics.get(f'{field_name}_effectiveness_ratio')
            if temp_eff_ratio is not None:
                print("\n📈 EFFECTIVENESS:")
                print(f"   Effectiveness Ratio:     {temp_eff_ratio:.4f}")
                print(f"   Original Unique:         {metrics.get(f'{field_name}_original_unique', 0)}")
                print(f"   Anonymized Unique:       {metrics.get(f'{field_name}_anonymized_unique', 0)}")
                print(f"   Null Increase:           {metrics.get(f'{field_name}_null_increase', 0):.4f}")
            
            # Display group statistics if applicable
            if 'group_count' in metrics:
                print("\n📊 GROUP-BASED SUPPRESSION:")
                print(f"   Group Count:             {metrics.get('group_count', 0):,}")
                print(f"   Group By Field:          {metrics.get('group_by_field', 'N/A')}")
                print(f"   Min Group Size:          {metrics.get('min_group_size', 0)}")
                
                if 'group_statistics_sample' in metrics:
                    sample = metrics['group_statistics_sample']
                    if sample:
                        print(f"\n   Group Statistics Sample (first 5):")
                        for i, (group, value) in enumerate(list(sample.items())[:5]):
                            print(f"      {group:30s}: {value}")
        
        except json.JSONDecodeError as e:
            print(f"❌ JSON decode error: {e}")
        except Exception as e:
            print(f"❌ Error reading metrics: {e}")

# 2. OUTPUT DATA (NEWEST)
print("\n\n2️⃣ OUTPUT DATA (First 10 rows):")
print("-" * 80)

output_dir = task_dir / 'output'
if not output_dir.exists():
    print(f"⚠️  Output directory not found: {output_dir}")
else:
    output_files = list(output_dir.glob('*.csv'))
    
    if not output_files:
        print("⚠️  No output files found")
    else:
        # Sort by modification time (newest first)
        output_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_output_file = output_files[0]
        
        # Show file info
        mtime = datetime.fromtimestamp(latest_output_file.stat().st_mtime)
        print(f"✓ Found {len(output_files)} output file(s)")
        print(f"📄 Loading LATEST: {latest_output_file.name}")
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_output_file.stat().st_size / 1024:.1f} KB\n")
        
        try:
            output_df = pd.read_csv(latest_output_file)
            if pd.api.types.is_datetime64_any_dtype(df[field_name]):
                output_df[field_name] = pd.to_datetime(output_df[field_name])
            
            print(f"📊 Shape: {output_df.shape[0]} rows × {output_df.shape[1]} columns")
            print(f"\n{output_df[[field_name]].head(10).to_string()}")
            
            # Show field distribution after suppression
            if field_name in output_df.columns:
                print(f"\n\n📊 {field_name.title()} Distribution (After Suppression):")
                print("-" * 80)
                safe_stats(output_df[field_name], label="After Suppression")
                
                print(f"\n📊 Comparison with Original:")
                print("-" * 80)

                # Show raw stats using safe_stats()
                print("\n🔹 Original field statistics:")
                safe_stats(df[field_name], label="Original")

                print("\n🔹 Processed field statistics:")
                safe_stats(output_df[field_name], label="Processed")

                col_orig = df[field_name]
                col_new  = output_df[field_name]

                print("\n🔍 Differences (Δ):")

                # Case 1 — Numeric column
                if pd.api.types.is_numeric_dtype(col_orig):

                    orig_mean = col_orig.mean()
                    orig_std  = col_orig.std()
                    orig_min  = col_orig.min()
                    orig_max  = col_orig.max()

                    new_mean = col_new.mean()
                    new_std  = col_new.std()
                    new_min  = col_new.min()
                    new_max  = col_new.max()

                    print(f"   Mean:     {orig_mean:.2f} → {new_mean:.2f} (Δ {new_mean - orig_mean:+.2f})")
                    print(f"   Std:      {orig_std:.2f} → {new_std:.2f} (Δ {new_std - orig_std:+.2f})")
                    print(f"   Min:      {orig_min:.2f} → {new_min:.2f} (Δ {new_min - orig_min:+.2f})")
                    print(f"   Max:      {orig_max:.2f} → {new_max:.2f} (Δ {new_max - orig_max:+.2f})")

                # Case 2 — Datetime column
                elif pd.api.types.is_datetime64_any_dtype(col_orig):

                    orig_min = col_orig.min()
                    orig_max = col_orig.max()

                    new_min  = col_new.min()
                    new_max  = col_new.max()

                    print(f"   Min:  {orig_min} → {new_min} (Δ {new_min - orig_min})")
                    print(f"   Max:  {orig_max} → {new_max} (Δ {new_max - orig_max})")

                # Case 3 — Strings / categorical → skip delta
                else:
                    print("   (Δ N/A — non-numeric, non-datetime column)")
            else:
                print(f"\n⚠️  Field '{field_name}' not found in output")
                print(f"Available columns: {', '.join(output_df.columns)}")
        
        except Exception as e:
            print(f"❌ Error reading output: {e}")

# 3. VISUALIZATIONS (NEWEST BATCH)
print("\n\n3️⃣ VISUALIZATIONS:")
print("-" * 80)

viz_dir = task_dir / 'visualizations'
if not viz_dir.exists():
    print(f"⚠️  Visualizations directory not found: {viz_dir}")
else:
    viz_files = list(viz_dir.glob('*.png'))
    
    if not viz_files:
        print("⚠️  No visualizations found")
    else:
        # Sort by modification time (newest first)
        viz_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        
        # Get timestamp from latest file to identify the batch
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            # Group files from same operation (within 10 seconds)
            time_threshold = 10  # seconds
            latest_viz_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < time_threshold
            ]
            
            # Sort batch by name for consistent ordering
            latest_viz_batch.sort(key=lambda x: x.name)
            
            # Show info
            latest_mtime = datetime.fromtimestamp(latest_time)
            print(f"✓ Found {len(viz_files)} total visualization(s)")
            print(f"📄 Showing LATEST batch: {len(latest_viz_batch)} files")
            print(f"   Created: {latest_mtime.strftime('%Y-%m-%d %H:%M:%S')}\n")
            
            # Display each visualization from latest batch
            for i, viz_file in enumerate(latest_viz_batch, 1):
                # Extract readable name from filename
                name_parts = viz_file.stem.split('_')
                # Create descriptive name
                if 'comparison' in viz_file.name.lower():
                    viz_name = f"{field_name.title()} - Before vs After Comparison"
                elif 'distribution' in viz_file.name.lower():
                    viz_name = f"{field_name.title()} - Distribution After Suppression"
                elif 'histogram' in viz_file.name.lower():
                    viz_name = f"{field_name.title()} - Histogram"
                else:
                    viz_name = viz_file.stem.replace('_', ' ').title()
                
                print(f"\n{i}. {viz_name}")
                print(f"   File: {viz_file.name}")
                print("-" * 60)
                try:
                    display(Image(filename=str(viz_file)))
                except Exception as e:
                    print(f"   ⚠️  Could not display: {e}")
        
        # Show info about older visualizations if any
        if len(viz_files) > len(latest_viz_batch):
            print(f"\n💡 Note: {len(viz_files) - len(latest_viz_batch)} older visualization(s) not shown")

# 4. SUPPRESSION DETAILS
print("\n\n4️⃣ SUPPRESSION DETAILS:")
print("-" * 80)

if output_files:
    try:
        # Safe comparison to avoid NaN != NaN evaluating as True
        suppressed_mask = (
            df[field_name].fillna("__NAN__") != result_df[field_name].fillna("__NAN__")
        )

        suppressed_records = df[suppressed_mask].copy()

        if len(suppressed_records) > 0:
            print(f"\n📋 Suppressed Records (showing first 10):")
            print("-" * 80)

            def fmt(v):
                return f"{v:.2f}" if pd.api.types.is_number(v) and pd.notna(v) else str(v)

            print(f"{'Original':<15} {'Suppressed':<15}")
            print("-" * 60)

            for idx in suppressed_records.index[:10]:
                orig_val = df.loc[idx, field_name]
                new_val = result_df.loc[idx, field_name]
                print(f"{fmt(orig_val):<15} {fmt(new_val):<15}")

            if len(suppressed_records) > 10:
                print(f"\n   ... and {len(suppressed_records) - 10} more suppressed records")

            print(f"\n📊 Suppressed Values Statistics:")
            print("-" * 80)

            orig_suppressed = df.loc[suppressed_mask, field_name]

            print(f"   Count: {orig_suppressed.notna().sum()}")

            safe_stats(orig_suppressed, label="Suppressed Values")
        else:
            print("\n✓ No cells were suppressed in this operation")
    except Exception as e:
        print(f"⚠️  Could not generate suppression details: {e}")
else:
    print("⚠️  No output data available for comparison")

print("\n\n" + "=" * 80)
print("✅ ARTIFACT REVIEW COMPLETE")
print("=" * 80)

## 🎯 Summary

**What you learned:**

✅ **Load data**: Read CSV from examples/data_examples/ with realistic patient measurements  
✅ **Setup environment**: TaskReporter, DataSource, ProgressTracker  
✅ **Configure operation**: Full production-style parameters with field config  
✅ **Execute**: Use operation.execute() with explicit configs  
✅ **Analyze results**: Review cell-level suppression metrics and artifacts

**Key takeaways:**
- `operation.execute()` handles end-to-end cell suppression
- Cell suppression provides fine-grained privacy protection at the value level
- Automatic outlier detection using IQR method identifies extreme values
- Suppression strategy (mean, median, mode, null, constant) determines replacement
- Field name is configurable via `create_config_kwargs()`
- Outliers are replaced while preserving overall data distribution
- All artifacts saved in structured directories

**Cell Suppression Benefits:**
- **Fine-grained control**: Suppress specific problematic values, not entire records
- **Privacy protection**: Removes outliers that could enable re-identification
- **Data utility**: Preserves majority of data for analysis
- **Flexible strategies**: Choose how to replace suppressed values
- **Automatic detection**: IQR and Z-score methods identify outliers

---

## 📚 Next Steps

**Ready for advanced features?** Try:

📘 **`02_cell_suppression_advanced.ipynb`** includes:
- All suppression strategies (null, mean, median, mode, constant, group_mean, group_mode)
- Group-based replacement strategies
- Multiple suppression triggers (outlier, rare, null, conditional)
- Z-score outlier detection method
- Rare value detection and handling
- Conditional suppression based on field values
- Custom suppression logic
- Performance optimization for large datasets

**Other Suppression Operations:**
- 📗 **`01_attribute_suppression_simple.ipynb`**: Remove entire columns
- 📙 **`01_record_suppression_simple.ipynb`**: Remove entire rows

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🐙 [GitHub Repository](https://github.com/DGT-Network/PAMOLA)